## Music Recommendation
### University of Virginia
### DS 5110: Big Data Systems
### Last Updated: March 18, 2026


---

#### Instructions

In this assignment, you will build an ALS recommendation algorithm based on user listening data from Autoscrobbler.

The data consists of: 
- user data (listeners)
- item data (songs)
- interaction data (user listens, which is implicit feedback).  

Additional details are in the `README` file. Be sure to review in advance.

Develop/test code to accomplish the tasks below, clearly showing the results and documenting your work.

You will use the **DataFrame API** for all work.

**TOTAL POINTS: 10**
***

#### Preprocessing Steps in order:

1) To reduce runtime, you will sample 20% of the interaction data from `user_artist_data.txt` using seed=314 (and use this seed throughout the lab)
2) Next, remove users that listened to fewer than 10 **artists** (you may ignore aliasing in this step)
3) Lastly, split the data: 60% for training / 20% for validation / 20% for testing

In [1]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# PREPROCESSING STEP 1
# To reduce runtime, you will sample 20% of the interaction data from user_artist_data.txt using seed=314 (and use this seed throughout the lab)
# ----------------------------------------------------------------------------------------------------------------------------------------------

# Before doing anything, create a Spark session.
# I added the .config settings per recommendation from Gemini to help reduce performance issues I was experiencing.
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("AutoscrobblerALS") \
    .config("spark.driver.memory", "32g") \
    .config("spark.sql.shuffle.partitions", "16") \
    .getOrCreate()

# Define the schema for the file to import.  
# -- I added this after reviewing the data file and running an initial import.
# -- Even though I saw the file had no headers, I didn't initially stop to think I needed to add them!  :)
from pyspark.sql.types import StructType, StructField, IntegerType
autoscrobbler_schema = StructType([
                                    StructField("userid", IntegerType()),
                                    StructField("artistid", IntegerType()),
                                    StructField("playcount", IntegerType()),
                                    ])
     

# Load the data.
# The columns are separated by a space, so need to include sep=" ".
df_user = spark.read.csv("/standard/ds7200-apt4c/large_datasets/user_artist_data.txt",  sep=" ", header=False, schema=autoscrobbler_schema)

# Check to see if any columns have null values.
df_user.createOrReplaceTempView("df_user")
df_user_null = spark.sql("""
    SELECT COUNT(*) AS RowsWithNulls
    FROM df_user
    WHERE 
        userid IS NULL 
        OR artistid IS NULL
        OR playcount IS NULL
""")
print(f"""There are {df_user_null.show()} rows with NULL values.""")

# Since no rows have null values, continue...
# Select a 20% sample from the data.
df_user_small = df_user.sample(fraction=0.20, seed=314)

# Confirm rows were imported.
print(f"""There are {df_user_small.count()} rows in the imported data set.""")

# Review a few records and the schema to make sure it was applied correctly.
# And it does, so yay!
df_user.show(5)
df_user.printSchema()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/11 21:31:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

+-------------+
|RowsWithNulls|
+-------------+
|            0|
+-------------+

There are None rows with NULL values.
There are 4859650 rows in the imported data set.
+-------+--------+---------+
| userid|artistid|playcount|
+-------+--------+---------+
|1000002|       1|       55|
|1000002| 1000006|       33|
|1000002| 1000007|        8|
|1000002| 1000009|      144|
|1000002| 1000010|      314|
+-------+--------+---------+
only showing top 5 rows
root
 |-- userid: integer (nullable = true)
 |-- artistid: integer (nullable = true)
 |-- playcount: integer (nullable = true)



In [2]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# PREPROCESSING STEP 2
# Next, remove users that listened to fewer than 10 artists (you may ignore aliasing in this step)
# ----------------------------------------------------------------------------------------------------------------------------------------------

# Remove users that listen to fewer than 10 artists using SQL.
# -- First register the DF as a temp view first.
df_user_small.createOrReplaceTempView("df_user_small")

# See how may users have listened to fewer than 10 artists.

# Select the columns needed for this analysis.
df_user_go = spark.sql("""
    SELECT *
    FROM df_user_small
    WHERE userid IN
        (
            SELECT userid
            FROM df_user_small
            GROUP BY userid
            HAVING COUNT(DISTINCT artistid) > 9
        )
""")

print(f"""There are {df_user_go.count()} rows after removing users that listened to less than 10 artists.""")

[Stage 14:====================================>                     (5 + 3) / 8]

There are 4684468 rows after removing users that listened to less than 10 artists.


In [3]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# PREPROCESSING STEP 3
# Lastly, split the data: 60% for training / 20% for validation / 20% for testing
# ----------------------------------------------------------------------------------------------------------------------------------------------
# Split the dataset.
trainDF, valDF, testDF = df_user_go.randomSplit([.6, .2, .2], seed=314)

# Cache the datasets for better downstream processing.
trainDF = trainDF.cache()
valDF = valDF.cache()
testDF = testDF.cache()

print(f"""There are {trainDF.count()} rows in the training set, {valDF.count()} rows in the validation set, and {testDF.count()} in the test set.""")

[Stage 55:======================================>                 (11 + 5) / 16]

There are 2808949 rows in the training set, 938515 rows in the validation set, and 937004 in the test set.


In [4]:
# TASK 1 (1 PT)
# Show the first row from the training set
trainDF.show(1)

# Result should remain consistent with seed=314.
#  |userid|artistid|playcount|
#  +------+--------+---------+
#  |   321|      40|        1|
#  +------+--------+---------+

+------+--------+---------+
|userid|artistid|playcount|
+------+--------+---------+
|   536|      30|        5|
+------+--------+---------+
only showing top 1 row


In [5]:
# TASK 2 (1 PT)
# Show the first row from the test set. We will refer to this user as Test User 1.
testDF.show(1)

# Result should remain consistent with seed=314.
#  +------+--------+---------+
#  |userid|artistid|playcount|
#  +------+--------+---------+
#  |   321|     357|        3|
#  +------+--------+---------+

+------+--------+---------+
|userid|artistid|playcount|
+------+--------+---------+
|   536|     341|        1|
+------+--------+---------+
only showing top 1 row


In [6]:
# ALS MODEL 1: Train the ALS implicit model on the training data (since the measurements are activity and not ratings)
# Use parameters: rank=8, maxIter=5, regParam=0.02, alpha=0.01

# The parameters above do not align to the old MLlib library, so using the newer ML library.
from pyspark.ml.recommendation import ALS

# Create the model specification using the parameters for ML.
als_spec = ALS(
    userCol="userid",
    itemCol="artistid",
    ratingCol="playcount",
    implicitPrefs=True,
    coldStartStrategy="drop",
    rank=8,
    maxIter=5,
    regParam=0.02,
    alpha=0.01
)

# Fit the model.
als_model = als_spec.fit(trainDF)

print("The model has been trained.")

26/10/11 21:31:53 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/10/11 21:31:53 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


The model has been trained.


In [7]:
# TASK 3 (2 PTS)
# Using ALS MODEL 1, make 10 recommendations for Test User 1
als_predictions = als_model.transform(testDF)

# Use SQL to get 10 predictions for Test User 1 (userid = 321).
als_predictions.createOrReplaceTempView("als_predictions")

# STEP 1: First, get the predictions for userid = 321.
test_user1 = spark.sql("""
    SELECT *
    FROM als_predictions
    WHERE 
        userid=321
        AND NOT ISNAN(prediction)
""")

# STEP 2: Then, get the top 10 predictions.
test_user1.createOrReplaceTempView("test_user1")
test_user1_top10 = spark.sql("""
    SELECT *
    FROM test_user1
    ORDER BY prediction DESC
    LIMIT 10
""")


test_user1_top10.show()

+------+--------+---------+-----------+
|userid|artistid|playcount| prediction|
+------+--------+---------+-----------+
|   321|     930|        1|0.085681416|
|   321|    2178|        3| 0.08560595|
|   321| 1275996|        1| 0.06496148|
|   321|    4951|      262|0.045232035|
|   321|    1991|       12|0.041772906|
|   321|     304|       19| 0.02334283|
|   321| 1028974|        3|0.019126914|
|   321| 1028717|        1|0.018754484|
|   321|    1365|        8|0.017662525|
|   321| 1233802|        7|0.017510727|
+------+--------+---------+-----------+



**TASK 4 (1 PT)**  
**Briefly explain Recall@K in the context of recommender systems**

Recall@k is like the concept of recall in class imbalance problems which is the number of True Positives (TP) that are correctly predicted, e.g., TP / (TP + FN).  What's different for recommendation problems is that the equivalent of True Positives (the numerator) is the top number (k) of the total predicted items of which the user did interact with and which the model correctly predicted as such, and the denominator is this number plus all the other items that the user interacted with.

In [8]:
# TASK 5 (1 PT)
# Calculate Recall@K for Test User 1

# This is asking, "Of all the artists that the user listened to, how many of the top-k predictions did the model correctly predict?"
# Mathematically, the formula is:
#
#              # artists the user listened to that are predicted in the top k
#  Recall@K = ----------------------------------------------------------------
#                     # all of the artists the user listened to


# One way to do this is with SQL.
# I'll stick with k=10 for this.

# This requires, querying the test data.
testDF.createOrReplaceTempView("testDF")

# STEP 1: DENOMINATOR.
# Get the total rows for userid = 321.
all_items = spark.sql("""
SELECT *
FROM testDF
WHERE userid = 321
""")

# STEP 2: NUMERATOR.
# Get the total of these that were in the predicted set; these are the true positives (tp).
# This part uses the top k=10 from TASK 3.
all_items.createOrReplaceTempView("all_items")
test_user1_top10.createOrReplaceTempView("test_user1_top10")
tp = spark.sql("""
SELECT *
FROM test_user1_top10 AS t
JOIN all_items AS a
ON t.artistid = a.artistid
""")

# STEP 3: CALCULATE the Recall@K.
tp.createOrReplaceTempView("tp")
recall_k = spark.sql("""
SELECT 
    ROUND(
            (SELECT COUNT(*) FROM tp) /
            (SELECT COUNT(*) FROM all_items)
            ,4
        ) AS RecallK
""")

# I had trouble extracting the value from the recall_k DataFrame so I had MS Copilot give me this syntax.
# I just don't do enough straight up Python coding for indexing to have sunk in.
print(f"The Recall@K for userid=321 is: {recall_k.collect()[0]['RecallK']}")

The Recall@K for userid=321 is: 0.2703


**Recall@K for my model at this point.**  
The above results show that the Recall@K for userid=321 is: 0.2703
The interpretation here is that out of all the artists listened to by userid=321 (in the test set), the modeled predicted about 27% of these artists in its top 10 recommendations.
That seems like a decent recommendation level.

In [9]:
# ALS MODEL 2: For this model, you will use hyperparameter tuning on the validation set to select the best values for
# (rank, maxIter, regParam, alpha) to maximize Recall@K

# Import the necessary library.
from pyspark.ml.tuning import ParamGridBuilder

# Build the hyperparameter tuning grid.
# I had Gemini help recommend values for these parameters.
# I added comments to explain my understanding of why to use these values.
paramGrid = (
        ParamGridBuilder()
    
        # Specify different number of hidden features to look for.
        .addGrid(als_spec.rank, [10, 20, 40])
    
        # Specify how many times to loop through the data.
        .addGrid(als_spec.maxIter, [10])
    
        # Specify the regularization penalty; 0.01 is flexible and 0.1 is stricter (less-overfitting).
        .addGrid(als_spec.regParam, [0.01, 0.1])
    
        # Specify how much weight is placed on the playcounts data; 1.0 a little weight, 10.0 a lot of weight.
        .addGrid(als_spec.alpha, [1.0, 10.0])
    
        # Build the tuning grid.
        .build()
    )

print("The paramGrid has been built.")

The paramGrid has been built.


In [10]:
# TASK 6 (2 PTS)
# Report the results including best values for (rank, maxIter, regParam, alpha) along with the Recall@K from the validation set

# Import the necessary libraries.
from pyspark.ml.tuning import CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator

# This evaluator will evaluate RMSE to tune parameters and then Recall@K will be calculated later on the validation set.
evaluator_spec = RegressionEvaluator(
    predictionCol="prediction",  # Use the ALS predictions.
    labelCol="playcount",        # Use the playcount data as the "truth" from the held-out data set for each CV fold.
    metricName="rmse",           # Specifies to optimize RMSE.
)

# Specify the cross validation approach.
crossval = CrossValidator(
    estimator=als_spec,
    estimatorParamMaps=paramGrid,
    evaluator= evaluator_spec,
    numFolds=2
)

print("The evaluator and crossval specs are built.")

The evaluator and crossval specs are built.


In [11]:
# THIS STEP TAKES SOME TIME SO IT IS PUT INTO ITS OWN CELL.

# Time this step.
import time
t0 = time.time()

# Run cross-validation on the validation data, leveraging parallel processing.
cvModel_val = crossval.setParallelism(4).fit(valDF)

# Show how much elapsed.
print("train time:", time.time() - t0)

train time: 73.20590496063232


**Training time experience.**  
I have seen training times as short as about 32 seconds and as long as almost 100 seconds.  
Sometimes though, this blows up and results in a connection error where the only way to resolve is restart the kernel, wait a while (I guess for the load on Rivanna to abate), and then re-run all the code again up to this point.

In [12]:
# Extract the best performing model and get the rank from it.
best_model_val = cvModel_val.bestModel
best_rank = best_model_val.rank

# To get the hyperparamters (maxIter, regParam, alpha) I asked Gemini for help getting the best values.
# I didn't see anything in the class material that explained how to get at these...I did read through almost everything!

# Find the index of the best validation metric (minimum RMSE).
best_idx = cvModel_val.avgMetrics.index(min(cvModel_val.avgMetrics))

# Retrieve the parameter map for that winning run.
best_params = cvModel_val.getEstimatorParamMaps()[best_idx]

# Create a clean dictionary mapping parameter names to their best values
best_params_dict = {param.name: value for param, value in best_params.items()}

# Assign them to explicit variables for later use in retraining the train + validation data.
best_maxIter = best_params_dict.get("maxIter")
best_regParam = best_params_dict.get("regParam")
best_alpha = best_params_dict.get("alpha")

# Dispaly the best values.
print("--- Best Hyperparameters Found ---")
print(f"rank = {best_rank}")
print(f"maxIter = {best_maxIter}")
print(f"regParam = {best_regParam}")
print(f"alpha = {best_alpha}")

print("\n--- Best RMSE ---")
print(f"Best Validation RMSE: {min(cvModel_val.avgMetrics):.4f}")

--- Best Hyperparameters Found ---
rank = 10
maxIter = 10
regParam = 0.01
alpha = 10.0

--- Best RMSE ---
Best Validation RMSE: 128.4979


**Best values.**

The above cell should return these results:

```text
--- Best Hyperparameters Found ---
rank = 10
maxIter = 10
regParam = 0.01
alpha = 10.0

--- Best RMSE ---
Best Validation RMSE: 128.4979

In [13]:
# Retrain the best configuration on train + validation data.

# Combine trainDF and valDF.
tvDF = trainDF.union(valDF)
tvDF = tvDF.cache()

# Update the model spec to use the best config.
als_spec_tuned = ALS(
    userCol="userid",
    itemCol="artistid",
    ratingCol="playcount",
    implicitPrefs=True,
    coldStartStrategy="drop",
    rank=best_model_val.rank,
    maxIter=best_maxIter,
    regParam=best_regParam,
    alpha=best_alpha
)

print("The tuned als_spec has been built.")

The tuned als_spec has been built.


In [14]:
# THIS STEP TAKES A FEW SECONDS...PUT INTO ITS OWN CELL IN CASE IT RUNS LONG LIKE THE CV MODEL TUNING.

# Time this step.
t0 = time.time()

# Fit the final model on the combined train and validation data.
als_model_tuned = als_spec_tuned.fit(tvDF) 

# Show how much elapsed.
print("train time:", time.time() - t0)

print("The final tuned model has been fit using the training and validation data.")

train time: 5.509636640548706
The final tuned model has been fit using the training and validation data.


**Training time experience.**  
This code tends to run in around 5-6 seconds.  

In [15]:
# TASK 7 (1 PT)
# Calculate Recall@K on the test set and report the value.

# For this, I will skip trying to integrate SQL and instead use PySpark.
# I had to get help from Gemini because I struggled a lot to piece this together.

# Import the necessary functions library.
from pyspark.sql import functions as F

# Get the unique set of userids.
test_users = testDF.select("userid").distinct()

# Generate the top 10 recommendations from this subset.
test_top10_recs = als_model_tuned.recommendForUserSubset(test_users, 10)

# Explode recommendations into a flat DataFrame of predicted items.
test_top10DF = test_top10_recs.select(
    F.col("userid"),
    F.explode(F.col("recommendations")).alias("rec")
).select(
    F.col("userid"),
    F.col("rec.artistid").alias("artistid")
)

# STEP 1: DENOMINATOR.
# Count the total number of artists for each user.
all_counts = testDF.groupBy("userid").agg(F.count("artistid").alias("total_actual"))

# STEP 2: NUMERATOR
# Count True Positives (tp) using an inner join.
# Get the total of these that were in the predicted set; these are the true positives (tp).
tp_all_count = test_top10DF.join(testDF, on=["userid", "artistid"], how="inner").groupBy("userid").agg(F.count("artistid").alias("tp_count"))

# STEP 3: CALCULATE the Recall@K.
recall_df = all_counts.join(tp_all_count, on="userid", how="left").na.fill(0, subset=["tp_count"]) \
    .withColumn("recall_k", F.col("tp_count") / F.col("total_actual"))

# Calculate the average over all users.
mean_recall_k = recall_df.agg(F.avg("recall_k")).collect()[0][0]

print(f"Mean Recall@K across the test set is: {mean_recall_k:.4f}")

Mean Recall@K across the test set is: 0.0302


**Test set Recall@K.**  
The prior cell should return something like this: *Mean Recall@K across the test set is: 0.0302*
This is better than the Recall@K obtained for userid=321 (0.2703).

In [17]:
# TASK 8 (1 PT)
# Calculate Recall@K for Test User 1 
# Justify the result, showing the recommendations and user activity. Did hyperparameter tuning seem to help for this user?

# For this, I will skip trying to integrate SQL and instead use PySpark.

# Import the necessary functions library.
from pyspark.sql import functions as F

# Get all the recommendations for userid = 321.
user321_recs = spark.createDataFrame([(321,)], ["userid"])

# Generate the top 10 recommendations from this subset.
user1_recs_top10 = als_model_tuned.recommendForUserSubset(user321_recs, 10)

# Explode recommendations into a flat DataFrame of predicted items.
top10DF = user1_recs_top10.select(
    F.col("userid"),
    F.explode(F.col("recommendations")).alias("rec")
).select(
    F.col("userid"),
    F.col("rec.artistid").alias("artistid")
)

# STEP 1: DENOMINATOR.
# Get all rows for userid = 321 from test data.
testDF_user321 = testDF.filter(F.col("userid") == 321)
testDF_user321_count = testDF_user321.count()

# STEP 2: NUMERATOR
# Count True Positives (tp) using an inner join.
# Get the total of these that were in the predicted set; these are the true positives (tp).
tp_count = top10DF.join(testDF_user321, on=["userid", "artistid"], how="inner").count()

# STEP 3: CALCULATE the Recall@K.
recall_k = tp_count / testDF_user321_count if testDF_user321_count > 0 else 0.0

print(f"The Recall@K for userid=321 is: {recall_k:.4f}")

The Recall@K for userid=321 is: 0.0270


In [18]:
# Show the recommendations and user activity.
print("--- Top 10 Artist Recommendations ---")
top10DF.show()

print("--- Actual Test Set Artists Listened To for User 321 ---")
testDF_user321.show()

--- Top 10 Artist Recommendations ---
+------+--------+
|userid|artistid|
+------+--------+
|   321|    4951|
|   321| 1011485|
|   321| 1001019|
|   321|    5717|
|   321| 1007027|
|   321| 1010373|
|   321| 1006845|
|   321|     122|
|   321|      71|
|   321|      53|
+------+--------+

--- Actual Test Set Artists Listened To for User 321 ---
+------+--------+---------+
|userid|artistid|playcount|
+------+--------+---------+
|   321|     304|       19|
|   321|     378|        7|
|   321|     716|        2|
|   321|     930|        1|
|   321|    1365|        8|
|   321|    1991|       12|
|   321|    2010|        4|
|   321|    2178|        3|
|   321|    2630|      101|
|   321|    4951|      262|
|   321| 1000782|       10|
|   321| 1003902|        3|
|   321| 1006231|       67|
|   321| 1008602|       28|
|   321| 1011306|        1|
|   321| 1011774|        2|
|   321| 1013287|        1|
|   321| 1014262|       42|
|   321| 1014332|        3|
|   321| 1017701|        3|
+------+

**Conclusion.**
The results above suggest the model only recommended one of the artists that userid=321 actually listened to (artistid = 4951).
Reviewing the top 10 recommendations and the artists in the test data set confirms this.
Per output above (visual inspection) there is just this one overlap between the predictions and the test data for userid=321.

Thus, the hyperparameter tuning did NOT improve recommendations for this user.